# NDC-classes: centrality and hyperedge communities
Upload this **.ipynb** through Colab's File → Upload notebook. Run all cells and upload **NDC-classes.tar.gz** when asked. No GPU is required. The last cell downloads the results ZIP.

This notebook calculates exact all-node dangling centrality (with progress and checkpoint output), hyperdegree, closeness, betweenness and weighted-projection eigenvector centrality. It applies the large-hyperedge/absolute-intersection branch of hyperedge percolation and reports sensitivity, memberships and descriptive associations. It is not a ground-truth community-recovery experiment.

**Representation:** static undirected hypergraph of unique unordered class-label sets. Repeated drug records are collapsed for topology; record frequencies are separately retained. Singleton hyperedges count toward hyperdegree, but create no pairwise connections. Timestamps are read and checked, not used in this static analysis.

**Important distinction:** removing a class label measures structural disruption, not therapeutic value or patient outcomes. Communities are not known clinical ground truth.

Sources: [dataset and required Benson et al. (2018) citation](https://www.cs.cornell.edu/~arb/data/NDC-classes/); [Kovács, Benedek & Palla (2025), hyperedge percolation](https://doi.org/10.1038/s41598-025-19974-9). Cite your original Dangling Centrality article for its provenance and explicitly specify the adaptation below.


In [ ]:
from pathlib import Path
from collections import Counter
import tarfile, time, json, hashlib, platform, zipfile
import numpy as np
import pandas as pd
import scipy, networkx as nx
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components
from scipy.sparse.linalg import eigsh
from scipy.stats import spearmanr
from IPython.display import display, Image

SEED = 42
PRIMARY_K, PRIMARY_S = 4, 3  # illustrative setting, not an optimized choice
SETTINGS = [(3,1),(3,2),(4,2),(4,3),(5,3),(5,4),(6,4),(6,5)]
OUT = Path('NDC_Hypergraph_results'); OUT.mkdir(exist_ok=True)
timings = []
def timed(name, fn):
    t = time.perf_counter(); result = fn()
    timings.append({'operation':name,'seconds':time.perf_counter()-t})
    return result
archive = Path('NDC-classes.tar.gz')
if not archive.exists():
    local = Path('upload/NDC-classes.tar.gz')
    if local.exists(): archive = local
    else:
        from google.colab import files
        uploaded = files.upload()
        candidates = [p for p in uploaded if p.endswith('.tar.gz')]
        if len(candidates)!=1: raise ValueError('Upload one NDC-classes .tar.gz archive.')
        archive = Path(candidates[0])
archive_hash = hashlib.sha256(archive.read_bytes()).hexdigest()


## 1. Validate the archive and construct the incidence matrix
Rows of H are class labels; columns are unique unordered hyperedges. The website's reported unique simplices may preserve different vertex orderings. We report both ordered-record uniqueness and unordered-set uniqueness instead of treating them as interchangeable.


In [ ]:
t = time.perf_counter()
with tarfile.open(archive, 'r:gz') as tar:
    def read_part(part):
        suffix = 'NDC-classes-'+part+'.txt'
        matches = [m for m in tar.getmembers() if m.isfile() and m.name.endswith(suffix)]
        if len(matches)!=1: raise ValueError('Missing or ambiguous file: '+suffix)
        return tar.extractfile(matches[0]).read().decode('utf-8')
    sizes = list(map(int, read_part('nverts').split()))
    flat = list(map(int, read_part('simplices').split()))
    timestamps = list(map(int, read_part('times').split()))
    labels = {int(line.split(maxsplit=1)[0]):line.split(maxsplit=1)[1]
              for line in read_part('node-labels').splitlines() if line.strip()}
    drug_labels = read_part('simplex-labels').splitlines()
assert sum(sizes)==len(flat) and len(sizes)==len(timestamps)
ordered=[]; unordered=[]; offset=0; duplicate_member_records=0
for size in sizes:
    e=tuple(flat[offset:offset+size]); offset+=size
    ordered.append(e); unordered.append(tuple(sorted(set(e))))
    duplicate_member_records += int(len(set(e))!=len(e))
frequency=Counter(unordered); edges=sorted(frequency)
vertices=sorted(set(flat)); n=len(vertices); index={v:i for i,v in enumerate(vertices)}
assert all(v in labels for v in vertices)
rows=[]; cols=[]
for j,e in enumerate(edges):
    for v in e: rows.append(index[v]); cols.append(j)
H=csr_matrix((np.ones(len(rows),dtype=np.int32),(rows,cols)),shape=(n,len(edges)))
edge_sizes=np.asarray(H.sum(axis=0)).ravel()
W=(H@H.T).astype(float); W.setdiag(0); W.eliminate_zeros()
A=W.copy(); A.data[:]=1
G=nx.from_scipy_sparse_array(A)
component_count, component_ids=connected_components(A,directed=False)
timings.append({'operation':'parse_and_construct','seconds':time.perf_counter()-t})
summary={'vertices':n,'timestamped_records':len(sizes),'unique_ordered_records':len(set(ordered)),
         'unique_unordered_hyperedges':len(edges),'incidences':H.nnz,
         'singleton_hyperedges':int((edge_sizes==1).sum()),'maximum_hyperedge_size':int(edge_sizes.max()),
         'projected_edges':G.number_of_edges(),'connected_components':component_count,
         'records_with_duplicate_members':duplicate_member_records,
         'simplex_label_lines':len(drug_labels),'incidence_density':H.nnz/(H.shape[0]*H.shape[1])}
display(pd.DataFrame(summary.items(),columns=['Statistic','Value']))
pd.DataFrame(summary.items(),columns=['Statistic','Value']).to_csv(OUT/'dataset_summary.csv',index=False)
pd.DataFrame({'hyperedge_id':range(1,len(edges)+1),'members':[';'.join(map(str,e)) for e in edges],
              'size':edge_sizes,'record_frequency':[frequency[e] for e in edges]}).to_csv(OUT/'hyperedges.csv',index=False)


## 2. Exact baseline centralities
- Hyperdegree: number of unique incident hyperedges, including singletons.
- Closeness: unweighted two-section distance, with NetworkX's disconnected-network correction.
- Betweenness: normalized unordered vertex-pair betweenness on the unweighted two-section. Alternative hyperedge-labelled paths are not counted separately.
- Eigenvector: leading eigenvector of W = HHᵀ − diag(hyperdegree), max-normalized. Weights count shared unique hyperedges; vertices themselves have no assigned weights. For disconnected graphs, the global eigenvector concentrates in components with maximal spectral radius. We check the spectral gap and residual; near-degeneracy prevents a unique global ranking.


In [ ]:
degree=timed('hyperdegree',lambda:np.asarray(H.sum(axis=1)).ravel())
closeness=timed('closeness_exact',lambda:nx.closeness_centrality(G,wf_improved=True))
betweenness=timed('betweenness_exact',lambda:nx.betweenness_centrality(G,normalized=True,weight=None))
def eigenvector():
    vals,vecs=eigsh(W,k=2,which='LA',tol=1e-10,v0=np.ones(n))
    order=np.argsort(vals); lam=vals[order[-1]]; x=vecs[:,order[-1]]
    if x.sum()<0: x=-x
    gap=float(lam-vals[order[-2]])
    residual=float(np.linalg.norm(W@x-lam*x)/max(abs(lam),1))
    if gap<1e-8*max(abs(lam),1):
        print('Near-degenerate leading eigenvalue: global ranking is not unique; scores left missing.')
        return np.full(n,np.nan),float(lam),gap,residual
    assert x.min()>-1e-6 and residual<1e-6
    x=np.maximum(x,0); x[x<1e-12]=0
    return x/x.max(),float(lam),gap,residual
eig,lam,gap,residual=timed('eigenvector_global',eigenvector)
results=pd.DataFrame({'node_id':vertices,'class_label':[labels[v] for v in vertices],
    'component_id':component_ids+1,'Hyperdegree':degree,
    'Closeness':[closeness[i] for i in range(n)],'Betweenness':[betweenness[i] for i in range(n)],'Eigenvector':eig})
print('Eigenvalue:',lam,'spectral gap:',gap,'relative residual:',residual)


## 3. Exact dangling centrality for every vertex
Φ(H) = Σ(i<j) 1/d(i,j), with unreachable-pair contributions zero and the diagonal excluded.

DC(v) = [Φ(H) − Φ(H⁻ᵛ)] / Φ(H).

H⁻ᵛ removes v from each incident hyperedge, retaining all other memberships; v becomes isolated. It does **not** delete entire incident hyperedges. With unit-cost vertex distances this equals node-isolation efficiency loss on the unweighted two-section, so it cannot distinguish hypergraphs having the same two-section.

For speed, recompute only v's original connected component: all other components are unchanged. Every vertex is evaluated exactly; this is not sampling. A checkpoint CSV is overwritten every 100 vertices. Runtime depends on hardware and component sizes.


In [ ]:
def phi(a):
    if a.shape[0]<2: return 0.0
    d=shortest_path(a,directed=False,unweighted=True,method='D')
    upper=d[np.triu_indices(len(d),1)]
    reachable=upper[np.isfinite(upper)&(upper>0)]
    return float((1/reachable).sum())
t=time.perf_counter()
components=[np.flatnonzero(component_ids==c) for c in range(component_count)]
blocks=[A[ids][:,ids].tocsr() for ids in components]
base=[phi(b) for b in blocks]; phi_original=sum(base)
dc=np.zeros(n); completed=0
for ids,b,p0 in zip(components,blocks,base):
    for local_i,global_i in enumerate(ids):
        if p0>0:
            changed=b.tolil(); changed[local_i,:]=0; changed[:,local_i]=0
            changed=changed.tocsr(); changed.eliminate_zeros()
            dc[global_i]=(p0-phi(changed))/phi_original
        completed+=1
        if completed%100==0:
            print(f'Exact dangling: {completed}/{n} vertices',flush=True)
            pd.DataFrame({'node_id':vertices,'Dangling_partial':dc}).to_csv(OUT/'dangling_partial.csv',index=False)
results['Dangling']=dc
assert np.isfinite(dc).all() and (dc>=-1e-10).all() and (dc<=1+1e-10).all()
timings.append({'operation':'dangling_exact_including_baselines','seconds':time.perf_counter()-t})
(OUT/'dangling_partial.csv').unlink(missing_ok=True)
print('Original communication strength:',phi_original)
display(results.sort_values('Dangling',ascending=False).head(15))


## 4. Hyperedge percolation and sensitivity
For each k,s: retain |e|≥k; connect retained hyperedges iff |eᵢ∩eⱼ|≥s; find connected components of that hyperedge-intersection graph; union their vertex sets. Remove duplicate and strictly contained community sets, as in the reference method. A community may consist of one retained hyperedge. Vertices in no retained community are unassigned.

This implements the paper's large-hyperedge branch only, not its small-hyperedge/relative-intersection branch. Hyperedges of size 1 or 2 are excluded at the selected k values, but remain in the original centrality network. Membership count is not a measure of clinical importance. Primary k=4,s=3 is illustrative, not selected to maximize correlation.


In [ ]:
def detect(k,s):
    assert 1<=s<=k-1
    selected=np.flatnonzero(edge_sizes>=k)
    if len(selected)==0: return [],np.zeros((n,0),dtype=bool),0
    h=H[:,selected]; overlap=(h.T@h).tocsr(); overlap.setdiag(0)
    overlap.data=(overlap.data>=s).astype(np.int32); overlap.eliminate_zeros()
    count,ids=connected_components(overlap,directed=False)
    candidates=set(frozenset().union(*(edges[selected[j]] for j in np.flatnonzero(ids==c))) for c in range(count))
    communities=[c for c in candidates if not any(c<other for other in candidates)]
    communities.sort(key=lambda c:(-len(c),tuple(sorted(c))))
    M=np.array([[v in c for c in communities] for v in vertices],dtype=bool)
    return communities,M,len(selected)
def rho(x,y):
    x=np.asarray(x); y=np.asarray(y); mask=np.isfinite(x)&np.isfinite(y)
    if mask.sum()<3 or np.ptp(x[mask])==0 or np.ptp(y[mask])==0: return np.nan
    return float(spearmanr(x[mask],y[mask]).statistic)
sensitivity=[]; all_memberships=[]; detected={}
for k,s in SETTINGS:
    t=time.perf_counter(); communities,M,retained=detect(k,s); elapsed=time.perf_counter()-t
    counts=M.sum(axis=1); detected[k,s]=(communities,M)
    sensitivity.append({'k':k,'s':s,'retained_hyperedges':retained,'communities':len(communities),
        'assigned_vertices':int((counts>0).sum()),'unassigned_vertices':int((counts==0).sum()),
        'overlap_vertices':int((counts>1).sum()),'overlap_fraction_all_vertices':float((counts>1).mean()),
        'largest_community':max(map(len,communities),default=0),
        'rho_dangling_membership_all':rho(dc,counts),
        'rho_dangling_membership_assigned':rho(dc[counts>0],counts[counts>0]),'seconds':elapsed})
    for j,c in enumerate(communities,1):
        all_memberships.extend({'k':k,'s':s,'community_id':j,'node_id':v} for v in sorted(c))
sensitivity=pd.DataFrame(sensitivity)
display(sensitivity)
sensitivity.to_csv(OUT/'parameter_sensitivity.csv',index=False)
pd.DataFrame(all_memberships).to_csv(OUT/'memberships_all_settings.csv',index=False)
communities,M=detected[PRIMARY_K,PRIMARY_S]
counts=M.sum(axis=1)
results['community_count']=counts
results['community_ids']=[';'.join(f'C{j+1}' for j in np.flatnonzero(row)) for row in M]
results['membership_status']=np.select([counts==0,counts==1],['unassigned','single'],default='overlap')
community_summary=pd.DataFrame([{'community_id':j+1,'vertices':len(c),
    'mean_dangling':float(dc[M[:,j]].mean()),'median_dangling':float(np.median(dc[M[:,j]]))} for j,c in enumerate(communities)])
community_summary.to_csv(OUT/'community_summary.csv',index=False)
results.to_csv(OUT/'node_centralities_and_communities.csv',index=False)
display(results.groupby('membership_status').Dangling.agg(['count','mean','median','min','max']))
display(results.sort_values('Dangling',ascending=False).head(15))


## 5. Figures and descriptive comparisons
Figures are saved as 300-dpi PNG and vector SVG. The incidence plot is the actual hypergraph membership matrix. Community membership is shown separately; no projected graph is presented as a native hypergraph drawing. Community numbers identify detected sets, not clinical categories.

Correlations describe associations on this one network. They are not independent observations proving accuracy, novelty or superiority. Report sensitivity and unassigned vertices alongside overlap. Further validation requires planted-community recovery and independently specified disruption experiments.


In [ ]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def savefig(fig,name):
    fig.tight_layout()
    for ext in ['png','svg']: fig.savefig(OUT/(name+'.'+ext),dpi=300,bbox_inches='tight')
    plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6)); ax.spy(H,markersize=.5,aspect='auto')
ax.set(title=f'NDC incidence: {n} class labels, {len(edges)} unique hyperedges',xlabel='Hyperedge column (sorted sets)',ylabel='Class-label row')
savefig(fig,'incidence_sparsity')
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(edge_sizes,bins=np.arange(.5,edge_sizes.max()+1.5),color='#287b8e')
axes[0].set(xlabel='Unique hyperedge size',ylabel='Count',title='Hyperedge cardinalities')
axes[1].bar(range(component_count),sorted(map(len,components),reverse=True),color='#287b8e')
axes[1].set(xlabel='Connected components (size order)',ylabel='Vertices',title='Unweighted two-section components')
savefig(fig,'structure')
measure_cols=['Hyperdegree','Closeness','Betweenness','Eigenvector','Dangling']
corr=results[measure_cols].corr(method='spearman'); corr.to_csv(OUT/'centrality_spearman.csv')
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm')
ax.set(xticks=range(5),yticks=range(5),xticklabels=measure_cols,yticklabels=measure_cols,title='Spearman correlations (descriptive)')
plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
for i in range(5):
    for j in range(5):ax.text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center',fontsize=9)
fig.colorbar(im,ax=ax); savefig(fig,'centrality_correlations')
fig,axes=plt.subplots(1,2,figsize=(13,6))
top=results.sort_values('Dangling',ascending=False).head(15).iloc[::-1]
color={'single':'#287b8e','overlap':'#bc5939','unassigned':'#999999'}
axes[0].barh(top.node_id.astype(str),top.Dangling,color=[color[x] for x in top.membership_status])
axes[0].set(xlabel='Dangling centrality',ylabel='Class-label ID (full names in CSV)',title='Top 15 class labels')
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=c,label=s) for s,c in color.items()],fontsize=8)
axes[1].scatter(counts,dc,s=12,alpha=.5,color='#287b8e')
axes[1].set(xlabel='Number of detected community memberships',ylabel='Dangling centrality',title=f'Primary setting k={PRIMARY_K}, s={PRIMARY_S}')
savefig(fig,'dangling_and_membership')
fig,ax=plt.subplots(figsize=(10,4))
setting_names=[f'k={r.k},s={r.s}' for r in sensitivity.itertuples()]
ax.bar(setting_names,sensitivity.communities,color='#287b8e'); ax.tick_params(axis='x',rotation=30)
ax.set(ylabel='Detected communities',title='Community-count parameter sensitivity')
savefig(fig,'community_sensitivity')
if M.shape[1]:
    order=sorted(range(n),key=lambda i:(-counts[i],-dc[i]))
    fig,ax=plt.subplots(figsize=(10,6)); ax.imshow(M[order],aspect='auto',interpolation='nearest',cmap='Blues',vmin=0,vmax=1)
    ax.set(xlabel='Community column (C1 onward)',ylabel='Vertices sorted by membership count, then dangling',title=f'Complete membership matrix: k={PRIMARY_K}, s={PRIMARY_S}')
    savefig(fig,'community_membership')


## 6. Runtime, complexity and downloadable results
Timings are one run on this machine, not a scalability benchmark. Construction and plotting are separate from centrality measurements. Let n be vertices, q be two-section edges, m be unique hyperedges, and L be incidences. Hyperdegree requires O(L) incidence counting; exact unweighted all-pairs BFS is O(n(n+q)); recomputing it per isolated vertex gives an O(n²(n+q)) reference bound. Component restriction can substantially reduce work. This implementation uses SciPy shortest paths, whose runtime also depends on its implementation. Dense distance storage is O(n_max²) for the largest component.

The hyperedge-intersection matrix can have O(m²) nonzeros; maximal-set filtering is also potentially quadratic in the number of candidate communities. Do not transfer this implementation unchanged to massive hyperedge collections. Eigenvector runtime depends on eigensolver iterations and the spectral gap. Empirical scalability needs several network sizes and repeated timings.


In [ ]:
runtime=pd.DataFrame(timings); display(runtime); runtime.to_csv(OUT/'runtime.csv',index=False)
metadata={'dataset_sha256':archive_hash,'representation':'static unique unordered hyperedges; singleton edges retained',
          'primary_k':PRIMARY_K,'primary_s':PRIMARY_S,'settings':SETTINGS,'seed':SEED,
          'phi_original':phi_original,'eigenvalue':lam,'spectral_gap':gap,'eigen_residual':residual,
          'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
                      'scipy':scipy.__version__,'networkx':nx.__version__},'dataset_summary':summary}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
(OUT/'README.txt').write_text('Static NDC unique-unordered-set analysis. See notebook for definitions and limitations.\n'
 'Primary community settings: k=4, s=3. All centralities use the original full hypergraph.\n'
 'Membership CSV includes all tested settings. Community labels have no ground-truth status.\n'
 'Dangling scores describe loss of class-label connectivity, not clinical effects.\n'
 'Runtime is a single local run, not a scalability estimate.\n')
zip_path=Path('NDC_Hypergraph_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print('Saved',zip_path)
try:
    from google.colab import files
except ImportError:
    print('Local run: ZIP is in the working directory.')
else:
    files.download(str(zip_path))
